# Mach-Zero Live Monitoring

Monitor live order book state, positions, and risk metrics via shared memory bridge.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import time
from bridge.shm_reader import MachZeroReader

In [ ]:
# Connect to shared memory
reader = MachZeroReader()
print(f'Connected: {reader.is_connected()}')
if reader.is_connected():
    print(f'Symbols: {reader.num_symbols()}')

In [ ]:
# Read all active symbols
if reader.is_connected():
    symbols = reader.read_all_active()
    for sym_id, state in symbols.items():
        print(f"Symbol {sym_id}:")
        print(f"  Bid: {state['bid_price']:.8f} x {state['bid_quantity']:.8f}")
        print(f"  Ask: {state['ask_price']:.8f} x {state['ask_quantity']:.8f}")
        print(f"  Last: {state['last_price']:.8f}")
        print(f"  Position: {state['position']:.8f}")
        print(f"  Orders: {state['order_count']}, Fills: {state['fill_count']}")
        print()

In [ ]:
# Live price ticker - polls shared memory
if reader.is_connected():
    SYMBOL_ID = 1  # BTCUSDT
    prices = []
    timestamps = []
    
    for i in range(50):
        state = reader.read_symbol(SYMBOL_ID)
        if state and state['last_price'] > 0:
            prices.append(state['last_price'])
            timestamps.append(time.time())
        time.sleep(0.1)
    
    print(f'Collected {len(prices)} price updates')
    if prices:
        print(f'Price range: {min(prices):.2f} - {max(prices):.2f}')

In [ ]:
# Plot prices if matplotlib is available
try:
    import matplotlib.pyplot as plt
    if prices:
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(range(len(prices)), prices, 'b-', linewidth=0.8)
        ax.set_xlabel('Tick')
        ax.set_ylabel('Price')
        ax.set_title('BTCUSDT Live Price')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
except ImportError:
    print('matplotlib not installed, skipping plot')